# Precious-Metals Statistical Arbitrage

## 1. Research question and decision standard

This notebook asks whether economically related precious-metals instruments exhibit a repeatable divergence/convergence effect that remains tradable after costs. It may conclude that no useful statistical arbitrage exists. Statistical significance is necessary, not sufficient, and none of the results guarantees future profit.

Verdicts are restricted to **Robust statistical-arbitrage evidence**, **Interesting but weak evidence**, or **No useful statistical arbitrage**.

## 2. Imports, deterministic configuration, and assumptions

Every historical estimate is point-in-time. The signal is observed first, execution occurs one session later, and P&L starts after execution. The base model charges 5 basis points per dollar of one-way turnover and 2% annualized short borrow.

The workflow follows the practical cautions in Michael Isichenko, *Quantitative Portfolio Management* (Wiley, 2021): prefer economically motivated relationships to brute-force pair mining, translate residual forecasts into actual tradable legs, model point-in-time information and costs, and avoid repeated P&L-driven tuning.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Callable
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.multitest import multipletests
from statsmodels.tsa.stattools import adfuller, coint

from ats.ticker import EquityTicker

RANDOM_SEED = 20260926
TRADABLE_TICKERS = ("GLD", "IAU", "SLV", "GDX", "GDXJ", "SIL", "SILJ")
FACTOR_TICKERS = ("SPY", "UUP", "IEF", "TIP")


@dataclass(frozen=True)
class ResearchConfig:
    discovery_fraction: float = 0.60
    validation_fraction: float = 0.20
    entry_z: float = 2.0
    exit_z: float = 0.5
    max_holding_days: int = 20
    transaction_cost_bps: float = 5.0
    annual_short_borrow: float = 0.02
    bootstrap_repetitions: int = 2_000
    bootstrap_block_length: int = 40
    horizons: tuple[int, ...] = (1, 5, 10, 20, 40)
    estimation_windows: tuple[int, ...] = (252, 504)
    accumulation_windows: tuple[int, ...] = (5, 10, 20)
    normalization_windows: tuple[int, ...] = (126, 252)
    robustness_entry_z: tuple[float, ...] = (1.5, 2.0, 2.5)
    robustness_exit_z: tuple[float, ...] = (0.0, 0.5)
    robustness_holding_days: tuple[int, ...] = (10, 20, 40)
    robustness_cost_bps: tuple[float, ...] = (2.0, 5.0, 10.0)
    robustness_borrow: tuple[float, ...] = (0.0, 0.02, 0.05)


@dataclass(frozen=True)
class SplitBoundaries:
    discovery_end: pd.Timestamp
    validation_end: pd.Timestamp
    test_end: pd.Timestamp


CONFIG = ResearchConfig()
np.random.seed(RANDOM_SEED)


def validate_price_panel(prices: pd.DataFrame) -> pd.DataFrame:
    '''Return sorted positive finite prices without inventing observations.'''
    if prices.index.has_duplicates:
        duplicates = prices.index[prices.index.duplicated()].unique()
        raise ValueError(f"duplicate price dates: {duplicates.tolist()}")
    clean = prices.copy()
    clean.index = pd.DatetimeIndex(clean.index)
    clean = clean.sort_index().astype(float)
    return clean.where(np.isfinite(clean) & clean.gt(0.0))


def simple_and_log_returns(prices: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Compute close-to-close returns while preserving gaps as missing.'''
    clean = validate_price_panel(prices)
    observed_pair = clean.notna() & clean.shift(1).notna()
    simple = clean.pct_change(fill_method=None).where(observed_pair)
    log = np.log(clean).diff().where(observed_pair)
    return simple, log


def fetch_price_panel(
    tickers: tuple[str, ...],
    ticker_factory: Callable[..., EquityTicker] = EquityTicker,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    '''Fetch maximum ATS histories and return a wide panel plus an audit table.'''
    series: dict[str, pd.Series] = {}
    audit_rows: list[dict[str, Any]] = []
    for symbol in tickers:
        try:
            ticker = ticker_factory(symbol).fetch_price_data(
                all_available_price_history=True
            )
            frame = ticker.price_data.select("date", "close").to_pandas()
            frame["date"] = pd.to_datetime(frame["date"])
            if frame.empty:
                raise ValueError("empty download")
            if frame["date"].duplicated().any():
                raise ValueError("duplicate dates")
            values = pd.Series(
                frame["close"].to_numpy(dtype=float),
                index=pd.DatetimeIndex(frame["date"]),
                name=symbol,
            ).sort_index()
            values = values.where(np.isfinite(values) & values.gt(0.0))
            series[symbol] = values
            audit_rows.append(
                {
                    "symbol": symbol,
                    "status": "ok",
                    "first_date": values.first_valid_index(),
                    "last_date": values.last_valid_index(),
                    "observations": int(values.notna().sum()),
                    "missing": int(values.isna().sum()),
                }
            )
        except Exception as exc:
            audit_rows.append(
                {
                    "symbol": symbol,
                    "status": f"excluded: {exc}",
                    "first_date": pd.NaT,
                    "last_date": pd.NaT,
                    "observations": 0,
                    "missing": 0,
                }
            )
    panel = pd.concat(series.values(), axis=1).sort_index() if series else pd.DataFrame()
    return panel, pd.DataFrame(audit_rows).set_index("symbol")


def chronological_boundaries(index: pd.DatetimeIndex) -> SplitBoundaries:
    '''Create ordered 60/20/20 date boundaries once for a candidate sample.'''
    dates = pd.DatetimeIndex(index).drop_duplicates().sort_values()
    if len(dates) < 5:
        raise ValueError("at least five observations are required for chronological splits")
    discovery_count = max(1, int(np.floor(len(dates) * CONFIG.discovery_fraction)))
    validation_count = max(1, int(np.floor(len(dates) * CONFIG.validation_fraction)))
    validation_end_pos = min(discovery_count + validation_count - 1, len(dates) - 2)
    return SplitBoundaries(
        discovery_end=dates[discovery_count - 1],
        validation_end=dates[validation_end_pos],
        test_end=dates[-1],
    )


def rolling_ols_lagged(
    y: pd.Series,
    x: pd.DataFrame,
    window: int,
    ridge_alpha: float = 0.0,
) -> tuple[pd.DataFrame, pd.Series]:
    '''Fit trailing regressions through t-1 and stamp coefficients/residual at t.'''
    if window < 2:
        raise ValueError("window must be at least two")
    aligned = pd.concat([y.rename("__target__"), x], axis=1).sort_index()
    columns = ["const", *x.columns]
    coefficients = pd.DataFrame(np.nan, index=aligned.index, columns=columns)
    residuals = pd.Series(np.nan, index=aligned.index, name="residual")
    penalty = np.diag([0.0, *([1.0] * x.shape[1])])

    for position in range(window, len(aligned)):
        history = aligned.iloc[position - window : position].dropna()
        if len(history) <= len(columns):
            continue
        design = np.column_stack(
            [np.ones(len(history)), history[x.columns].to_numpy(dtype=float)]
        )
        target = history["__target__"].to_numpy(dtype=float)
        condition = np.linalg.cond(design)
        if ridge_alpha > 0.0:
            estimate = np.linalg.solve(
                design.T @ design + ridge_alpha * penalty,
                design.T @ target,
            )
        elif not np.isfinite(condition) or condition > 1e12:
            continue
        else:
            estimate = np.linalg.lstsq(design, target, rcond=None)[0]
        timestamp = aligned.index[position]
        coefficients.loc[timestamp] = estimate
        current = aligned.iloc[position]
        if current.notna().all():
            current_design = np.r_[1.0, current[x.columns].to_numpy(dtype=float)]
            residuals.loc[timestamp] = float(current["__target__"] - current_design @ estimate)
    return coefficients, residuals


def lagged_robust_zscore(values: pd.Series, window: int) -> pd.Series:
    '''Standardize each value from a median and scale ending at t-1.'''
    if window < 3:
        raise ValueError("window must be at least three")
    result = pd.Series(np.nan, index=values.index, name="zscore")
    numeric = values.astype(float)
    for position in range(window, len(numeric)):
        history = numeric.iloc[position - window : position].dropna()
        current = numeric.iloc[position]
        if len(history) < max(3, window // 2) or not np.isfinite(current):
            continue
        center = float(history.median())
        mad = float((history - center).abs().median())
        scale = 1.4826 * mad
        if not np.isfinite(scale) or scale <= 0.0:
            scale = float(history.std(ddof=1))
        if np.isfinite(scale) and scale > 0.0:
            result.iloc[position] = (current - center) / scale
    return result

## 3. ATS API audit and data acquisition

Data are fetched below through `EquityTicker`; adjusted closes and actual observation dates are retained.

## 4. Data quality and chronological splits

Each candidate uses chronological 60/20/20 discovery, validation, and untouched-test partitions.

## 5. Reusable leakage-safe research functions

The tagged cells below contain the estimators, inference, and backtest machinery exercised by focused tests.

## 6. Deterministic self-tests

Assertions here complement the focused pytest suite with notebook-visible invariants.

## 7. Discovery diagnostics for candidate families

Only economically constrained cointegration, factor-residual, and rolling-PCA candidates are considered.

## 8. Convergence shape, robust inference, and rejection table

Forward convergence is tested at several horizons with overlap-aware inference and discovery-family multiplicity control.

## 9. Frozen strategy specification

The base entry, exit, holding period, sizing, timing, and cost assumptions are frozen before the untouched test is exposed.

## 10. Validation results and local robustness

Validation can reject the discovery candidate; robustness is diagnostic and never feeds parameter selection.

## 11. Untouched out-of-sample test

At most one validated primary candidate reaches this section.

## 12. Trading performance, regimes, and failure modes

Gross and net results, turnover, borrow, tails, drawdowns, subperiods, and lagged regimes are reported separately.

## 13. Concise final research summary

The executed notebook ends with the exact divergence, evidence, rule, out-of-sample result, robustness, limitations, and permitted verdict.